* Original article: https://towardsdatascience.com/how-to-handle-missing-data-8646b18db0d4
* or https://cran.r-project.org/web/packages/finalfit/vignettes/missing.html
* Another article: https://insightsoftware.com/blog/how-to-handle-missing-data-values-while-data-cleaning/

1. **Missing Completely at Random (MCAR)**: The fact that a certain value is missing has nothing to do with its hypothetical value and with the values of other variables.
    
    For example, survey data is missing because someone could not make it to an appointment, or an administrator misplaces the test results he is supposed to enter into the computer. The reason for the missing values is unrelated to the data in the dataset.


2. **Missing conditionally at Random (MAR)**: Missing at random means that the propensity for a data point to be missing is not related to the missing data, but it is related to some of the observed data. 
    
    The reason the data is missing in a column can be explained by the data in other columns. For example, a school student who scores above the cutoff is typically given a grade. So, a missing grade for a student can be explained by the column that has scores below the cutoff.


3. **Missing not at Random (MNAR)**: Two possible reasons are that the missing value depends on the hypothetical value (e.g. People with high salaries generally do not want to reveal their incomes in surveys) or missing value is dependent on some other variable’s value (e.g. Let’s assume that females generally don’t want to reveal their ages! Here the missing value in age variable is impacted by gender variable)

    There is a correlation between the missing values and the actual income. The missing values are not dependent on other variables in the dataset.
    

![](https://preview.redd.it/sfogalc384z81.jpg?auto=webp&v=enabled&s=a2f57c812e4f6935e05e09c7cfa4ac90f434477b)

*Data imputation (completing the dataset, filling in) does not necessarily gives better result, but might be useful for dealing with the data itself.*

<img src="https://miro.medium.com/max/700/1*_RA3mCS30Pr0vUxbp25Yxw.png">

### Maybe recover values? Fill up missing values from other sources?

A, Algorithms such as random forest and KNN are robust in dealing with missing values.

B, Deal with missing data on our own:
* **delete the rows with missing values**: Typically, any row which has a missing value in any cell gets deleted. However, this often means many rows will get removed, leading to loss of information and data. Therefore, this method is typically not used when there are few data samples.

* **impute the missing data**: This can be based solely on information in the column that has missing values, or it can be based on other columns present in the dataset. 

* use **classification** or **regression models** to predict missing values.

In [4]:
import pandas as pd
import sklearn

**A more detailed version overview**
- **Goal:** Clean/filter data and impute missing values robustly.
- **Approach:** Detect/filter bad data first, then impute using methods matched to data type, missingness mechanism, and downstream model.



**1) Data filtering (pre-imputation)**
- **Assess Missingness:** 
  - Tools: `pandas.isna()`, `missingno.matrix()`, `sklearn.impute` diagnostics.
  - When: Always first to decide MCAR/MAR/MNAR.
- **Outlier detection**
  - Algorithms: Z-score, Interquartile Range (IQR) method, robust Mahalanobis, isolation forest.
  - Tools: `scipy.stats`, `numpy`, `sklearn.ensemble.IsolationForest`
  - Pros/cons: Z-score simple but sensitive to non-normal; IQR robust for skewed; IsolationForest works for multivariate.

In [16]:
# Example (IQR):

# Example Data
data = {'scores': [74, 88, 78, 90, 94, 90, 84, 90, 98, 80, 50, 150]}
df = pd.DataFrame(data)

# 1. Calculate Q1 and Q3
Q1 = df['scores'].quantile(0.25)
Q3 = df['scores'].quantile(0.75)

# 2. Calculate IQR
IQR = Q3 - Q1

# 3. Define Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. Filter
df_filtered = df[(df['scores'] >= lower_bound) & (df['scores'] <= upper_bound)]

print(f"Original Shape: {df.shape}")
print(f"Filtered Shape: {df_filtered.shape}")
print(f"Outliers: {df['scores'].loc[~df['scores'].isin(df_filtered['scores'])]}")



Original Shape: (12, 1)
Filtered Shape: (10, 1)
Outliers: 10     50
11    150
Name: scores, dtype: int64


In [14]:
IF = sklearn.ensemble.IsolationForest()
IF.fit(df['scores'].values.reshape(-1, 1))
df['outlier'] = IF.predict(df['scores'].values.reshape(-1, 1))
print(f"Outliers: {df['scores'][df['outlier'] == -1]}")

Outliers: 0      74
10     50
11    150
Name: scores, dtype: int64


- **Noise reduction / smoothing**
  - Algorithms: rolling median/mean, Savitzky–Golay, low-pass filters.
  - Tools: `pandas.Series.rolling()`, `scipy.signal.savgol_filter`
- **Record removal vs. correction**
  - Drop rows only if missingness/outlier rate is high or MCAR; otherwise impute.



**2) Simple imputation (fast baseline)**
- **Mean/Median/Mode**
  - Tools: `pandas.fillna()`, `sklearn.impute.SimpleImputer(strategy='mean'|'median'|'most_frequent')`
  - Use when missing fraction small and MCAR, or as baseline.
  - Example (mean):
    ```
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='mean')
    X_imputed = imp.fit_transform(X)
    ```
- **Constant / indicator**
  - Use sentinel values or add missingness indicator columns for models that can use them.



**3) Nearest-neighbor imputation**
- **KNNImputer**
  - Tools: `sklearn.impute.KNNImputer`
  - Pros: preserves local structure; works for mixed missingness patterns.
  - Cons: expensive on large datasets; needs scaling.
  - Preprocessing: scale numeric features; encode categoricals (e.g., one-hot or ordinal with care).
  - Example:


In [18]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_knn = imputer.fit_transform(df[['scores']])

X_knn

array([[ 74.],
       [ 88.],
       [ 78.],
       [ 90.],
       [ 94.],
       [ 90.],
       [ 84.],
       [ 90.],
       [ 98.],
       [ 80.],
       [ 50.],
       [150.]])


**4) Model-based & regression imputation**
- **Single regression imputation**
  - Fit regression of feature with missing values on other features, predict missing rows.
  - Tools: `sklearn.linear_model.LinearRegression`, `DecisionTreeRegressor`
  

In [ ]:
# Example:
train = df[df['y'].notna()]; predict = df[df['y'].isna()]
model.fit(train[cols], train['y'])
df.loc[df['y'].isna(), 'y'] = model.predict(predict[cols])


- **Iterative Imputer / MICE (Multiple Imputation by Chained Equations)**
  - Tools: `sklearn.impute.IterativeImputer` (approx MICE), `statsmodels` / `fancyimpute` / `miceforest` for more options.
  - Pros: accounts for joint distributions, can use different estimators per column.
  - Cons: slower; requires careful convergence checks.


In [ ]:
# Example:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imputer = IterativeImputer(random_state=0)
X_iter = imputer.fit_transform(X)

In [19]:
import numpy as np
from sklearn.impute import KNNImputer

df = pd.DataFrame({
    "age": [25, 30, np.nan, 40, 35],
    "salary": [50000, np.nan, 62000, 70000, 65000],
    "score": [80, 85, 88, np.nan, 90]
})

imputer = KNNImputer(n_neighbors=2)
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print("Original:")
print(df)
print("\nImputed:")
print(df_imputed)

Original:
    age   salary  score
0  25.0  50000.0   80.0
1  30.0      NaN   85.0
2   NaN  62000.0   88.0
3  40.0  70000.0    NaN
4  35.0  65000.0   90.0

Imputed:
    age   salary  score
0  25.0  50000.0   80.0
1  30.0  56000.0   85.0
2  32.5  62000.0   88.0
3  40.0  70000.0   87.5
4  35.0  65000.0   90.0



**5) Probabilistic / EM approaches**
- **Expectation–Maximization (EM) / Gaussian Mixture**
  - Tools: `sklearn.mixture.GaussianMixture`, `fancyimpute` implementations
  - Use when data approx multivariate normal or mixture; yields uncertainty estimates.
  - Example sketch:
    - Fit GMM to observed data, use conditional expectations to impute missing entries (requires custom impl or library).
- **Multiple imputation + pooling**
  - Libraries: `statsmodels.imputation.mice`, `miceforest`, `impyute`
  - Workflow: create multiple completed datasets, run analysis on each, pool estimates using Rubin's rules.



**6) Time-series specific methods**
- **Forward/Backward Fill**
  - Tools: `pandas.fillna(method='ffill'|'bfill')`
  - Use for sensor gaps or stateful series.
- **Interpolation**
  - Tools: `pandas.Series.interpolate(method='linear'|'time'|'spline')`
- **Kalman / state-space**
  - Tools: `statsmodels.tsa.statespace`, `pmdarima`, `pykalman`
  - Use for stochastic processes where temporal dynamics matter.
  - Example (linear interpolation):
    ```python
    df['y'] = df['y'].interpolate(method='time')
    ```



**7) Categorical data imputation**
- **Mode / constant**
  - Tools: `SimpleImputer(strategy='most_frequent')`
- **Predictive models**
  - Use classification models to predict missing category.
  - Example:
    ```python
    clf.fit(train[cols], train['cat'])
    df.loc[df['cat'].isna(),'cat'] = clf.predict(predict[cols])
    ```
- **Target encoding / embedding-aware imputations**
  - For high-cardinality categories, consider target/mean encoding or learned embeddings (deep models).



**8) Deep learning approaches**
- **Autoencoders, denoising AE, VAE**
  - Tools: `PyTorch`, `TensorFlow/Keras`, libraries like `DataWig` for mixed types.
  - Pros: model complex nonlinear relationships; can handle high-dimensional data.
  - Cons: need lots of data, careful training and validation.
- **Example idea:** train a denoising autoencoder to reconstruct full records; use masked inputs to predict missing features.



**9) Validation and diagnostics**
- **Holdout & simulation**
  - Mask a known subset of observed data, impute, compare using RMSE, MAE, classification accuracy.
- **Downstream impact**
  - Validate imputation by training downstream model (e.g., classifier/regressor) and comparing performance across imputation strategies.
- **Uncertainty**
  - Prefer multiple imputation when inference requires valid uncertainty estimates.



**10) Practical recommendations (quick guide)**
- If missingness < 5% and MCAR: `SimpleImputer` (mean/median) + indicator.
- If structure/local similarity matters: `KNNImputer`.
- If relationships between variables are strong: `IterativeImputer`/MICE.
- For time-series: prefer interpolation/Kalman/state-space methods.
- For inference and reporting: use Multiple Imputation (MICE) and pool estimates.
- For production/pipelines: implement reproducible imputation with training-only fit (save imputer), and log missingness features.
